In [ ]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

class Fuzzifier:
    def __init__(self, mf_type='gaussian', n_partitions=5):
        self.mf_type = mf_type
        self.n_partitions = n_partitions
        self.params = {}

    def fit(self, X):
        for col in X.columns:
            min_val = np.percentile(X[col].dropna(), 1)
            max_val = np.percentile(X[col].dropna(), 99)
            if min_val == max_val: 
                min_val, max_val = X[col].min(), X[col].max()
                if min_val == max_val: max_val += 1e-9 
            self.params[col] = np.linspace(min_val, max_val, self.n_partitions)

    def transform(self, X):
        X_fuzzy = pd.DataFrame(index=X.index)
        ramos_nomes = ['M_Baixo', 'Baixo', 'Medio', 'Alto', 'M_Alto'] # Nomenclatura clara
        
        for col in X.columns:
            centers = self.params.get(col)
            w = centers[1] - centers[0] if len(centers) > 1 else 1.0
            
            for i, c in enumerate(centers):
                col_name = f"{col}_{ramos_nomes[i]}"
                # Gaussian Membership Function
                mf = np.exp(-0.5 * ((X[col] - c) / (w/1.5))**2)
                X_fuzzy[col_name] = mf
        return X_fuzzy

df_orig = pd.read_csv('dataset.csv', parse_dates=['datetime'])
features_originais = ['SMA_3', 'EMA_3', 'SMA_5', 'EMA_5', 'std_close3', 'std_open3', 'ADXR', 'Bollinger_Norm']

fuzzifier = Fuzzifier(mf_type='gaussian', n_partitions=5)

X_orig = df_orig[features_originais]
fuzzifier.fit(X_orig)
X_fuzzy = fuzzifier.transform(X_orig)

X_fuzzy['trend'] = df_orig['trend']
X_fuzzy['datetime'] = df_orig['datetime']

X_fuzzy.to_csv('dataset_fuzzy_gaussian_5.csv', index=False)
print(f"Sucesso! Dataset gerado com {X_fuzzy.shape[1] - 2} features fuzzy (40 pertinências).")
print("Arquivo 'dataset_fuzzy_gaussian_5.csv' salvo.")

In [ ]:
%pip install xgboost skrebate

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from skrebate import ReliefF

df = pd.read_csv('dataset_fuzzy_gaussian_5.csv', parse_dates=['datetime'])
features = [col for col in df.columns if col not in ['datetime', 'trend']]

X_train = df[df['datetime'] < '2024-04-01'][features]
y_train = df[df['datetime'] < '2024-04-01']['trend']

rankings = pd.DataFrame(index=features)

print("Iniciando avaliação de importância de atributos. Isso pode levar alguns minutos...\n")

print("-> Calculando Random Forest Importance...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rankings['RandomForest'] = rf.feature_importances_

# 2. Information Gain (Mutual Information)
print("-> Calculando Information Gain (MI)...")
rankings['InfoGain'] = mutual_info_classif(X_train, y_train, random_state=42)

# 3. ReliefF
print("-> Calculando ReliefF...")
relieff = ReliefF(n_features_to_select=10, n_neighbors=50) # Vizinhos ajustados para reduzir tempo
relieff.fit(X_train.values, y_train.values)
rankings['ReliefF'] = relieff.feature_importances_

# 4. XGBoost
print("-> Calculando XGBoost Importance...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
rankings['XGBoost'] = xgb.feature_importances_

# 5. ANOVA F-Value
print("-> Calculando ANOVA F-value...")
f_values, _ = f_classif(X_train, y_train)
rankings['ANOVA'] = f_values

# 6. Lasso Regularization (L1)
print("-> Calculando Lasso (L1 Penalty)...")
lasso = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=500)
lasso.fit(X_train, y_train)
rankings['Lasso'] = np.abs(lasso.coef_[0])

# 7. Permutation Importance (Usando Random Forest como base)
print("-> Calculando Permutation Importance...")
perm_imp = permutation_importance(rf, X_train, y_train, n_repeats=5, random_state=42)
rankings['Permutation'] = perm_imp.importances_mean

# Gerar as ordens de importância (Ranking 1 = mais importante)
ordenacao = pd.DataFrame(index=features)
for col in rankings.columns:
    # Ordena descrescentemente e retorna a ordem do índice (rank)
    ordenacao[col] = rankings[col].rank(method='min', ascending=False)

print("\nConcluído! Rankings processados.")
display(ordenacao.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
import warnings

warnings.filterwarnings('ignore')

class PreFuzzyDecisionTree:
    def __init__(self, max_depth=8, min_samples_leaf=10):
        self.tree = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf, 
                                           criterion='entropy', random_state=42)
    def fit(self, X_f, y): 
        self.tree.fit(X_f, y)
    def predict_proba(self, X_f): 
        return self.tree.predict_proba(X_f)
    
    def predict_proba_soft(self, X_f):
        t = self.tree.tree_
        n_s = X_f.shape[0]
        res = np.zeros((n_s, self.tree.n_classes_))
        q = [(0, np.ones(n_s))]
        X_a = X_f.values
        while q:
            node, weights = q.pop(0)
            if np.all(weights == 0): continue
            if t.children_left[node] == t.children_right[node]:
                v = t.value[node][0]
                res += weights[:, np.newaxis] * (v / v.sum())
            else:
                feat = t.feature[node]
                # Como os dados já vieram pré-fuzzificados [0,1], isso flui perfeitamente
                mu = np.clip(X_a[:, feat], 0, 1)
                q.append((t.children_right[node], weights * mu))
                q.append((t.children_left[node], weights * (1-mu)))
        return res

class PreFuzzyRandomForest:
    def __init__(self, n_estimators=100, voting='MWLFUS', max_depth=8, min_samples_leaf=10):
        self.n_estimators = n_estimators
        self.voting = voting
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.trees, self.features_per_tree, self.tree_weights = [], [], []

    def fit(self, X_f, y):
        self.classes_ = np.unique(y)
        n_s, n_f = X_f.shape
        # Garante que pelo menos 1 feature seja selecionada quando 'k' for muito pequeno no SFS
        m_f = max(1, int(np.sqrt(n_f))) 
        
        for _ in range(self.n_estimators):
            idx = np.random.choice(n_s, size=n_s, replace=True)
            oob = np.array(list(set(range(n_s)) - set(idx)))
            feats = np.random.choice(n_f, size=m_f, replace=False)
            self.features_per_tree.append(feats)
            
            fdt = PreFuzzyDecisionTree(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf)
            fdt.fit(X_f.iloc[idx, feats], y.iloc[idx])
            
            if len(oob) > 0:
                preds_o = fdt.tree.predict(X_f.iloc[oob, feats])
                acc = accuracy_score(y.iloc[oob], preds_o)
                self.tree_weights.append(acc)
                if self.voting == 'MWLFUS':
                    fdt.error_tree = (NearestNeighbors(n_neighbors=5).fit(X_f.iloc[oob, feats]), 
                                      (preds_o != y.iloc[oob]).astype(int).values)
            else: 
                self.tree_weights.append(1.0)
            self.trees.append(fdt)

    def predict(self, X_f):
        votes = np.zeros((X_f.shape[0], len(self.classes_)))
        for i, tree in enumerate(self.trees):
            X_sub = X_f.iloc[:, self.features_per_tree[i]]
            p = tree.predict_proba_soft(X_sub) if self.voting == 'SOFT_ROUTING' else tree.predict_proba(X_sub)
            if self.voting == 'SMI': w = 1.0
            elif self.voting == 'MWLT': w = self.tree_weights[i]
            elif self.voting == 'MWLFUS':
                knn, err = tree.error_tree
                _, ids = knn.kneighbors(X_sub)
                w = (1.0 - err[ids].mean(axis=1))[:, np.newaxis]
            else: w = 1.0
            votes += p * w
        return self.classes_[np.argmax(votes, axis=1)]

# Carregar o dataset principal com datas para respeitar a divisão temporal
df = pd.read_csv('dataset_fuzzy_gaussian_5.csv', parse_dates=['datetime'])

# Obter as listas ordenadas de features geradas no Bloco 2
metodos_selecao = ordenacao.columns
resultados_sfs = {metodo: [] for metodo in metodos_selecao}

# SFS: Testar do top 1 até o top 40 (adicionando de 2 em 2)
step = 2 
n_features_teste = list(range(1, 41, step))
if 40 not in n_features_teste: n_features_teste.append(40)

print(f"{'='*60}")
print(f"SFS - FUZZY RANDOM FOREST (Voting: MWLFUS)")
print(f"{'='*60}")

# Filtragem de segurança: Garantir que só pegamos as colunas fuzzy (ignorando trend/datetime)
X_train_full = df[df['datetime'] < '2024-04-01']
y_train_full = X_train_full['trend']

X_test_full = df[df['datetime'] >= '2024-04-01']
y_test_full = X_test_full['trend']

for metodo in metodos_selecao:
    print(f"\n>> Avaliando Ranking: [{metodo}]")
    colunas_fuzzificadas_ordenadas = ordenacao[metodo].sort_values().index.tolist()
    
    for k in n_features_teste:
        features_subconjunto = colunas_fuzzificadas_ordenadas[:k]
        
        # Cortando os datasets apenas com as K melhores features daquele método
        X_train_k = X_train_full[features_subconjunto]
        X_test_k  = X_test_full[features_subconjunto]
        
        # Treinando a SUA Fuzzy Random Forest autêntica
        frf_sfs = PreFuzzyRandomForest(n_estimators=100, voting='MWLFUS', max_depth=8, min_samples_leaf=10)
        frf_sfs.fit(X_train_k, y_train_full)
        
        preds = frf_sfs.predict(X_test_k)
        acc = accuracy_score(y_test_full, preds)
        
        resultados_sfs[metodo].append(acc)
        print(f"   Top {k:>2} features -> Acc: {acc:.4f}")

plt.figure(figsize=(14, 8))
cores = cm.tab10.colors

for i, metodo in enumerate(metodos_selecao):
    plt.plot(n_features_teste, np.array(resultados_sfs[metodo]) * 100, 
             marker='o', linestyle='-', linewidth=2, color=cores[i], label=metodo)

plt.title('Seleção Sequencial (SFS) com Fuzzy Random Forest (MWLFUS)', fontsize=15, fontweight='bold')
plt.xlabel('Número Acumulado de Features (Top K)', fontsize=12, fontweight='bold')
plt.ylabel('Acurácia na Base de Teste (%)', fontsize=12, fontweight='bold')
plt.xticks(n_features_teste)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Método de Ranking', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Salvar e exibir
plt.savefig('sfs_frf_mwlfus.png', dpi=300)
plt.show()

# Encontrar o Pico Global (A melhor combinação absoluta de Método e Quantidade)
max_acc = 0
best_method = ""
best_k = 0

for m in metodos_selecao:
    m_max = max(resultados_sfs[m])
    k_idx = resultados_sfs[m].index(m_max)
    k_val = n_features_teste[k_idx]
    
    if m_max > max_acc:
        max_acc = m_max
        best_method = m
        best_k = k_val

print("\n" + "="*60)
print(f"Método de Seleção Campeão : {best_method}")
print(f"Ponto de Corte (Top K)    : {best_k} features")
print(f"Acurácia de Teste Máxima  : {max_acc*100:.2f}%")
print("="*60)